# Regulus — end-to-end walkthrough

One notebook for the whole project: ingest **real** regulatory texts, look up the
provisions that apply to a plain-language issue, build the **regulatory knowledge
graph**, and surface **cited cross-framework references**.

- **Phase 1** — ingest (EU AI Act · NIST AI RMF) → issue → applicable provisions (with citations).
- **Phase 2** — regulatory knowledge graph + cited crosswalks → issue → provisions annotated with risks and cross-framework references.

Regulus is built on the [Geometric Knowledge Network (GKN)](https://github.com/minw0607/geometric_knowledge_network). The default retriever is TF-IDF (no API keys); set `REGULUS_RETRIEVER=embedding` in `.env` for embedding-quality retrieval.

## 1. Setup

Makes `regulus` importable and falls back to a local GKN checkout (sibling folder)
if the `geometric_knowledge_network` package is not installed — so this runs in any
kernel with the base scientific stack, without a `pip install`.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))
try:
    import geometric_knowledge_network  # noqa: F401
except ModuleNotFoundError:
    gkn_src = Path.cwd().parent.parent / 'geometric_knowledge_network' / 'src'
    if gkn_src.exists():
        sys.path.insert(0, str(gkn_src))
        print('Using local GKN checkout at', gkn_src)
    else:
        raise ModuleNotFoundError("Install GKN: pip install git+https://github.com/minw0607/geometric_knowledge_network")

from regulus.config import RegulusConfig
from regulus.standards_loader import StandardsLoader
from regulus.lookup import RegulusLookup
from regulus.crosswalk import load_crosswalks
from regulus.graph import RegulusGraphBuilder, graph_summary
from regulus.graph_lookup import RegulusGraphLookup

config = RegulusConfig()
print('cache dir:', config.cache_dir)
print('retriever:', config.retriever, '| top_k:', config.top_k)

## 2. Ingest real standards  ·  *Phase 1*

Downloads and caches the source documents on first run (EUR-Lex HTML, NIST PDF),
then parses them into citable `Provision` records. Later runs load from cache.

> Parsing the NIST PDF needs `pypdf`. If it is missing, NIST is skipped with a
> warning and the EU AI Act still loads — `pip install pypdf` to include NIST.

In [ ]:
from collections import Counter

provisions = StandardsLoader(config).load(framework_ids=['eu_ai_act', 'nist_ai_rmf'])
print(dict(Counter(p.framework_name for p in provisions)))

for p in provisions[:2]:
    print('\n', p.citation())
    print('  source:', p.source_url)
    print('  text:', p.text[:180], '...')

## 3. Baseline lookup: issue → applicable provisions  ·  *Phase 1*

Semantic retrieval over the provision corpus. Try your own issue below.

In [ ]:
lookup = RegulusLookup(provisions, config)
print(f'Indexed {len(provisions)} provisions as {len(lookup.chunks)} chunks.\n')

for issue in [
    'Our credit model was deployed without testing for demographic bias across protected groups.',
    'We run real-time facial recognition in public spaces to assist law enforcement.',
]:
    print('=' * 100)
    print('ISSUE:', issue, '\n')
    for i, r in enumerate(lookup.search(issue, top_k=4), 1):
        print(f'{i}. [{r.score:.3f}] {r.provision.citation()}')
        print(f'     source: {r.provision.source_url}')
    print()

## 4. Build the regulatory knowledge graph  ·  *Phase 2*

Frameworks, provisions, and risk categories, linked by `CONTAINS`, `ADDRESSES`
(keyword-derived, low-confidence), and **`CROSSWALK`** edges. Crosswalks come only
from the curated, cited table `data/crosswalks/crosswalks.csv` — never inferred.

In [ ]:
crosswalks = load_crosswalks()
graph = RegulusGraphBuilder().build(provisions, crosswalks)
print(f'Loaded {len(crosswalks)} crosswalk rows.\n')
for key, count in graph_summary(graph).items():
    print(f'  {key:24s} {count}')

## 5. Crosswalk-aware lookup  ·  *Phase 2*

The payoff: each applicable provision is annotated with the **risks** it addresses
and its **cited cross-framework references**.

In [ ]:
gl = RegulusGraphLookup(provisions, config)

for issue in [
    'Our credit model was deployed without testing for demographic bias.',
    'We have no post-deployment monitoring for our high-risk AI system.',
]:
    print('=' * 100)
    print('ISSUE:', issue, '\n')
    for r in gl.search(issue, top_k=3):
        print(f'* {r.provision.citation()}   (score {r.score:.3f})')
        if r.risks:
            print(f'    risks: {", ".join(r.risks)}')
        for cx in r.crosswalks:
            print(f'    -> {cx.provision.citation()}  [{cx.relation}]')
            print(f'        rationale: {cx.rationale}')
            print(f'        source: {cx.source}')
    print()

## 6. What's next

- **Phase 3** — use GKN's multi-hop retriever + path explainer to return the full **evidence path** (issue → provision → crosswalk → provision).
- **Phase 4** — an LLM interpretation layer producing a structured, cited answer (risks · standards · cross-refs · guidance).
- **Phase 5** — a "submit an issue" UI and an evaluation benchmark (issue → expected-standards, crosswalk accuracy).

To improve/extend the crosswalks, edit `data/crosswalks/crosswalks.csv` — replace the seed rows with authoritative mappings and add more frameworks. For embedding-quality retrieval, set `REGULUS_RETRIEVER=embedding` in `.env`.